# S4 - ML distribuido con Spark MLlib (Regresion)

**Actividad:** construir el notebook `04_ml_distribuido_regresion_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), entrenando y comparando modelos de regresion distribuida con Spark MLlib sobre un dataset real de sensores ambientales, y reportando metricas iniciales (RMSE, R2, MAE).


## 1. El dataset: sensores ambientales reales

`campo_electrico_particionado/` es la salida particionada en Parquet que construye S3b (Calidad de datos y particionamiento, segundo caso de uso) integrando tres fuentes reales de sensores (campo electrico, campo magnetico, variables ambientales), medidas minuto a minuto: 184 538 filas completas en las 9 variables, sin nulos, particionadas por mes (`AnioMes`).

El objetivo de hoy: estimar `Valor_CE` (campo electrico) a partir de las otras 8 variables medidas en el mismo instante — un problema de regresion multivariable clasico, sin ningun componente temporal (no se usa el minuto anterior ni el siguiente; cada fila es una observacion independiente). La version con horizonte de prediccion (estimar el valor del minuto **siguiente** usando historial) es contenido de S10 (Series de tiempo e inferencia en streaming), no de hoy.


## 2. Crear la `SparkSession`


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


In [ ]:
ORIGEN_DATOS = "/opt/s03b-calidad-campo-electrico/artifacts/campo_electrico_particionado"
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"


## 3. Cargar el dataset (salida particionada de S3b) y explorar las 9 variables

Se lee directo con `spark.read.parquet()` -- un Parquet particionado ya trae su propio esquema, no hace falta declararlo a mano como con un CSV (S3b, paso 2). `AnioMes` reaparece como columna aunque no esta guardada dentro de ningun archivo: Spark la reconstruye a partir del nombre de la carpeta de particion (mismo *partition discovery* de S3).


In [ ]:
VARIABLES_9 = [
    "Valor_CE", "Valor_CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "Rain", "SolarRad.", "UVIndex",
]

df = spark.read.parquet(ORIGEN_DATOS)

df.printSchema()
print(f"Filas: {df.count():,}")
df.describe(VARIABLES_9).show()


## 4. Preparar el vector de predictores (`VectorAssembler`)

Spark MLlib no acepta columnas sueltas como entrada de un modelo — necesita una sola columna vectorial que agrupe todos los predictores. `VectorAssembler` hace exactamente eso: toma N columnas numericas y las combina en una columna `features` de tipo `Vector`. `Valor_CE` queda fuera de los predictores: es la columna objetivo (`label`), no un dato de entrada.


In [ ]:
from pyspark.ml.feature import VectorAssembler

PREDICTORES = [v for v in VARIABLES_9 if v != "Valor_CE"]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", "Valor_CE")

df_ml.show(5, truncate=False)


## 5. Dividir en entrenamiento y prueba

Division aleatoria simple (80/20), no cronologica — a diferencia de una tarea de pronostico (S10), aqui cada fila es una observacion independiente, sin orden temporal que preservar.


In [ ]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


## 6. Entrenar un modelo base: `LinearRegression`


In [ ]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="Valor_CE")
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


## 7. Evaluar el modelo (RMSE, R2, MAE)


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select("Valor_CE", "prediction").show(5)

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(
            labelCol="Valor_CE", predictionCol="prediction", metricName=metrica
        )
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(predicciones_base, "LinearRegression base")


## 8. Comparar configuraciones basicas (`regParam` / `elasticNetParam`)

El silabo pide comparar configuraciones basicas, no solo entrenar un unico modelo. `regParam` controla cuanto se penaliza la magnitud de los coeficientes (regularizacion); `elasticNetParam` mezcla penalizacion L1 (Lasso, `=1.0`) y L2 (Ridge, `=0.0`). Se prueban tres configuraciones simples, sin busqueda exhaustiva de hiperparametros (eso queda fuera del alcance de esta sesion):


In [ ]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="Valor_CE",
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


## 9. Comparar con un segundo algoritmo: `RandomForestRegressor`

`LinearRegression` asume una relacion lineal entre predictores y objetivo. `RandomForestRegressor` no — captura relaciones no lineales e interacciones entre variables sin necesitar ese supuesto. Comparar ambas familias (lineal vs. arboles) es la forma mas basica de saber si la relacion real es, de entrada, aproximadamente lineal.


In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features", labelCol="Valor_CE",
    numTrees=50, maxDepth=8, seed=42,
)
modelo_rf = rf.fit(df_train)
predicciones_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(predicciones_rf, "Random Forest")


## 10. Comparacion final y seleccion


In [ ]:
comparacion_final = pd.DataFrame(comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
])[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")


## 11. Guardar el modelo seleccionado

Elige, en base a la Tabla de la celda anterior, cual configuracion tuvo el mejor RMSE en tu propia corrida, y guardala. El nombre de variable `modelo_ganador` de abajo asume que fue el modelo de Random Forest — ajustalo segun tu resultado real.


In [ ]:
modelo_ganador = modelo_rf  # ajusta esta linea segun tu propio resultado (celda anterior)

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ce_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_ce_regresion")


## 12. Documentar hallazgos y responder preguntas de reflexion

Agrega celdas markdown breves debajo de cada bloque de codigo (secciones 6-10) explicando que hiciste y que observaste — es la base directa de la evidencia tecnica que armaras en 4.3.1.

**Reflexion tecnica breve** (5 a 8 lineas): ?que diferencia de RMSE encontraste entre la configuracion sin regularizacion y la de Random Forest? ?por que `VectorAssembler` es un paso obligatorio en Spark MLlib y no en scikit-learn? ?que significaria un R2 cercano a 0 para este problema, y tu resultado se acerco a eso o se alejo?
